# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [48]:
#Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestRegressor,AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier 
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV



from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

SEED = 1

In [8]:
import sys
print(sys.executable)

/Users/juan/Documents/Ironhack/Python/ML module/lab-ensemble/.venv/bin/python


In [9]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


Now perform the same as before:
- Feature Scaling
- Feature Selection


In order to do that, first I will perform an EDA (same steps as previous lab with this dataset)

In [10]:
# Info about the dataset
spaceship.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [11]:
#Check for null values
spaceship.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [12]:
# For this exercise, because we have such low amount of null values, we will drop rows containing any missing value. 

spaceship.dropna(inplace=True)

In [13]:
# Cabin is too granular - transform it in order to obtain {'A', 'B', 'C', 'D', 'E', 'F', 'G', 'T'}

spaceship["Cabin"] = spaceship["Cabin"].apply(lambda x: x.split("/")[0])
spaceship.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [14]:
# Drop PassengerId and Name
spaceship = spaceship.drop(columns=['PassengerId', 'Name'])

**Perform Train Test Split**

In [15]:
# Select the features (X) and the target (y)
X = spaceship[['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP','RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']]
y = spaceship['Transported']

# ============================================================
# Train / Test Split
# ============================================================

# Split the data into training (80%) and testing (20%)
# stratify=y keeps the same proportion of the target classes
# in both the training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (5284, 11)
X_test shape: (1322, 11)


In [16]:
# ============================================================
# Separate Categorical and Numerical Features
# ============================================================

# Categorical variables that need One-Hot Encoding
categorical_cols = [
    'HomePlanet',
    'CryoSleep',
    'Cabin',
    'Destination',
    'VIP'
]

# Numerical variables that will be kept as numbers
numeric_cols = [
    'Age',
    'RoomService',
    'FoodCourt',
    'ShoppingMall',
    'Spa',
    'VRDeck'
]

In [17]:
# ============================================================
# One-Hot Encoding
# ============================================================

# Create the encoder
# drop='first' avoids multicollinearity
# handle_unknown='ignore' prevents errors if new categories
# appear in the test set

encoder = OneHotEncoder(
    sparse_output=False,
    drop='first',
    handle_unknown='ignore'
)

# Learn the categories from the training data and transform it
X_train_cat = encoder.fit_transform(X_train[categorical_cols])

# Apply the same transformation to the test data
X_test_cat = encoder.transform(X_test[categorical_cols])

# Retrieve the names of the newly created columns
encoded_cols = encoder.get_feature_names_out(categorical_cols)

# Convert the encoded arrays back into DataFrames
# (this keeps the column names and original row indexes)

X_train_cat = pd.DataFrame(
    X_train_cat,
    columns=encoded_cols,
    index=X_train.index
)

X_test_cat = pd.DataFrame(
    X_test_cat,
    columns=encoded_cols,
    index=X_test.index
)


In [18]:
# ============================================================
# Keep the Numerical Features
# ============================================================

# Extract the numerical columns from the original dataset

X_train_num = X_train[numeric_cols]
X_test_num = X_test[numeric_cols]


# ============================================================
# Combine Numerical and Encoded Features
# ============================================================

# Join the numerical variables with the encoded categorical ones

X_train_final = pd.concat(
    [X_train_num, X_train_cat],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat],
    axis=1
)

In [19]:
# ============================================================
# Feature Scaling
# ============================================================

# KNN is distance-based, so all features should be on a similar scale

scaler = StandardScaler()

# Return a pandas DataFrame instead of a NumPy array
scaler.set_output(transform="pandas")

# Compute the mean and standard deviation using ONLY the training set
# and standardize the training data

X_train_final = scaler.fit_transform(X_train_final)

# Apply the same scaling to the test data

X_test_final = scaler.transform(X_test_final)


# ============================================================
# Check the Result
# ============================================================

print(X_train_final.head())
print(X_train_final.shape)
print(X_test_final.shape)

           Age  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck  \
6608 -0.062685    -0.347862  -0.194775     -0.298866  1.109513  1.010481   
7683  0.696693    -0.318144   1.929981     -0.298866  0.101711  0.197504   
8083 -0.545925    -0.347862  -0.293514     -0.298866 -0.278132 -0.264883   
4011  1.801242    -0.347862  -0.137999     -0.297178 -0.278132  0.229701   
6539 -0.269788    -0.347862   0.587738     -0.298866  1.059890 -0.264883   

      HomePlanet_Europa  HomePlanet_Mars  CryoSleep_True   Cabin_B   Cabin_C  \
6608           1.704522        -0.513627       -0.742856 -0.328263  3.192948   
7683           1.704522        -0.513627       -0.742856 -0.328263  3.192948   
8083          -0.586675         1.946939        1.346155 -0.328263 -0.313190   
4011          -0.586675        -0.513627       -0.742856 -0.328263 -0.313190   
6539          -0.586675        -0.513627       -0.742856 -0.328263 -0.313190   

       Cabin_D   Cabin_E   Cabin_F  Cabin_G   Cabin_T  \
6608 

**Model Selection** - now you will try to apply different ensemble methods in order to get a better model

- Bagging and Pasting

In [20]:
bagging_clf = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    random_state=SEED
)

bagging_clf.fit(X_train_final, y_train)

,"estimator estimator: object, default=NoneThe base estimator to fit on random subsets of the dataset.If None, then the base estimator is a:class:`~sklearn.tree.DecisionTreeClassifier`... versionadded:: 1.2 `base_estimator` was renamed to `estimator`.",DecisionTreeC...andom_state=1)
,"n_estimators n_estimators: int, default=10The number of base estimators in the ensemble.",100
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random resampling of the original dataset(sample wise and feature wise).If the base estimator accepts a `random_state` attribute, a differentseed is generated for each instance in the ensemble.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",1
,"max_samples max_samples: int or float, default=NoneThe number of samples to draw from X to train each base estimator (withreplacement by default, see `bootstrap` for more details).- If None, then draw `X.shape[0]` samples irrespective of `sample_weight`.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` unweighted samples or `max_samples * sample_weight.sum()` weighted samples.",None
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator (without replacement by default, see `bootstrap_features` for moredetails).- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.",1.0
,"bootstrap bootstrap: bool, default=TrueWhether samples are drawn with replacement. If False, sampling withoutreplacement is performed. If fitting with `sample_weight`, it isstrongly recommended to choose True, as only drawing with replacementwill ensure the expected frequency semantics of `sample_weight`.",True
,"bootstrap_features bootstrap_features: bool, default=FalseWhether features are drawn with replacement.",False
,"oob_score oob_score: bool, default=FalseWhether to use out-of-bag samples to estimatethe generalization error. Only available if bootstrap=True.",False
,"warm_start warm_start: bool, default=FalseWhen set to True, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fita whole new ensemble. See :term:`the Glossary <warm_start>`... versionadded:: 0.17 *warm_start* constructor parameter.",False
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for both :meth:`fit` and:meth:`predict`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",0


In [21]:
# ============================================================
# Make Predictions
# ============================================================

# Predict the target values for the training set
y_pred_train = bagging_clf.predict(X_train_final)

# Predict the target values for the test set
y_pred_test = bagging_clf.predict(X_test_final)


# ============================================================
# Evaluate the Model
# ============================================================

# Compute the training accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {accuracy_train * 100:.2f}%")

# Compute the testing accuracy
accuracy_test = accuracy_score(y_test, y_pred_test)
print(f"Testing Accuracy: {accuracy_test * 100:.2f}%")

print()

# Display precision, recall, F1-score and support
print(classification_report(
    y_test,
    y_pred_test,
    target_names=['Not Transported', 'Transported']
))

Training Accuracy: 94.13%
Testing Accuracy: 77.91%

                 precision    recall  f1-score   support

Not Transported       0.77      0.80      0.78       656
    Transported       0.79      0.76      0.78       666

       accuracy                           0.78      1322
      macro avg       0.78      0.78      0.78      1322
   weighted avg       0.78      0.78      0.78      1322



The Bagging Classifier achieved a training accuracy of 94.13% and a testing accuracy of 77.91%. The gap between the training and testing accuracy suggests that the model exhibits some degree of overfitting, as it performs considerably better on the training data than on unseen test data. On the test set, the model achieved precision, recall, and F1-scores of approximately 0.78 for both the Transported and Not Transported classes, indicating balanced performance and no significant bias toward either class. Overall, the Bagging Classifier provides a reasonable level of predictive accuracy but could benefit from further tuning to improve its generalization performance.

- Random Forests

In [22]:
X_train_final.head()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,HomePlanet_Europa,HomePlanet_Mars,CryoSleep_True,Cabin_B,Cabin_C,Cabin_D,Cabin_E,Cabin_F,Cabin_G,Cabin_T,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,VIP_True
6608,-0.062685,-0.347862,-0.194775,-0.298866,1.109513,1.010481,1.704522,-0.513627,-0.742856,-0.328263,3.192948,-0.250502,-0.336339,-0.690808,-0.65253,-0.019459,-0.318264,-1.507930,-0.158191
7683,0.696693,-0.318144,1.929981,-0.298866,0.101711,0.197504,1.704522,-0.513627,-0.742856,-0.328263,3.192948,-0.250502,-0.336339,-0.690808,-0.65253,-0.019459,-0.318264,0.663161,-0.158191
8083,-0.545925,-0.347862,-0.293514,-0.298866,-0.278132,-0.264883,-0.586675,1.946939,1.346155,-0.328263,-0.313190,-0.250502,2.973189,-0.690808,-0.65253,-0.019459,-0.318264,0.663161,-0.158191
4011,1.801242,-0.347862,-0.137999,-0.297178,-0.278132,0.229701,-0.586675,-0.513627,-0.742856,-0.328263,-0.313190,-0.250502,-0.336339,1.447580,-0.65253,-0.019459,-0.318264,0.663161,-0.158191
6539,-0.269788,-0.347862,0.587738,-0.298866,1.059890,-0.264883,-0.586675,-0.513627,-0.742856,-0.328263,-0.313190,-0.250502,-0.336339,1.447580,-0.65253,-0.019459,3.142042,-1.507930,-0.158191


In [23]:
# Step 4 - Model Training: Random Forest Classifier
# -------------------------------------------------

# No max_depth cap, same approach as the Regressor
rf_clf = RandomForestClassifier(
    n_estimators=100,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features=2,      # sqrt rule for Classification: sqrt(6) = 2.45 --> rounded to 2
    random_state=SEED)

rf_clf.fit(X_train_final, y_train)
print("Model Trained!")

Model Trained!


In [24]:
# ============================================================
# Make Predictions
# ============================================================

# Predict the target values for the training set
y_pred_train = rf_clf.predict(X_train_final)

# Predict the target values for the test set
y_pred_test = rf_clf.predict(X_test_final)


# ============================================================
# Evaluate the Model
# ============================================================

# Compute the training accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {accuracy_train * 100:.2f}%")

# Compute the testing accuracy
accuracy_test = accuracy_score(y_test, y_pred_test)
print(f"Testing Accuracy: {accuracy_test * 100:.2f}%")

print()

# Display precision, recall, F1-score and support
print(classification_report(
    y_test,
    y_pred_test,
    target_names=['Not Transported', 'Transported']
))

Training Accuracy: 88.91%
Testing Accuracy: 77.99%

                 precision    recall  f1-score   support

Not Transported       0.76      0.80      0.78       656
    Transported       0.80      0.76      0.78       666

       accuracy                           0.78      1322
      macro avg       0.78      0.78      0.78      1322
   weighted avg       0.78      0.78      0.78      1322



The Random Forest classifier achieved a training accuracy of 88.91% and a testing accuracy of 77.99%. Compared with the Bagging Classifier, the Random Forest exhibited a smaller gap between training and testing accuracy, indicating better generalization and reduced overfitting. On the test set, the model achieved an overall accuracy of 77.99%, with precision, recall, and F1-scores of approximately 0.78 for both the Transported and Not Transported classes. Overall, the Random Forest provided balanced performance and represented the strongest ensemble model evaluated due to its comparable predictive accuracy and improved generalization.

- Gradient Boosting

In [25]:
gb_clf = GradientBoostingClassifier(    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=SEED
)

gb_clf.fit(X_train_final, y_train)

print("Model Trained!")

Model Trained!


In [26]:
# ============================================================
# Make Predictions
# ============================================================

# Predict the target values for the training set
y_pred_train = gb_clf.predict(X_train_final)

# Predict the target values for the test set
y_pred_test = gb_clf.predict(X_test_final)


# ============================================================
# Evaluate the Model
# ============================================================

# Compute the training accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {accuracy_train * 100:.2f}%")

# Compute the testing accuracy
accuracy_test = accuracy_score(y_test, y_pred_test)
print(f"Testing Accuracy: {accuracy_test * 100:.2f}%")

print()

# Display precision, recall, F1-score and support
print(classification_report(
    y_test,
    y_pred_test,
    target_names=['Not Transported', 'Transported']
))

Training Accuracy: 82.25%
Testing Accuracy: 79.05%

                 precision    recall  f1-score   support

Not Transported       0.81      0.76      0.78       656
    Transported       0.78      0.82      0.80       666

       accuracy                           0.79      1322
      macro avg       0.79      0.79      0.79      1322
   weighted avg       0.79      0.79      0.79      1322



The Gradient Boosting classifier achieved a training accuracy of 82.25% and a testing accuracy of 79.05%. The relatively small difference between the training and testing accuracies indicates that the model generalized well to unseen data and showed limited evidence of overfitting. On the test set, the model achieved an overall accuracy of 79.05%, with balanced precision, recall, and F1-scores of approximately 0.79 for both the Not Transported and Transported classes. Overall, the Gradient Boosting classifier demonstrated the strongest predictive performance among the ensemble models evaluated, providing the highest testing accuracy while maintaining balanced classification across both classes.

- Adaptive Boosting

In [27]:
ada_clf = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=100,
    learning_rate=1.0,
    random_state=SEED
)
ada_clf.fit(X_train_final, y_train)

print("Model Trained!")

Model Trained!


In [28]:
# ============================================================
# Make Predictions
# ============================================================

# Predict the target values for the training set
y_pred_train = ada_clf.predict(X_train_final)

# Predict the target values for the test set
y_pred_test = ada_clf.predict(X_test_final)


# ============================================================
# Evaluate the Model
# ============================================================

# Compute the training accuracy
accuracy_train = accuracy_score(y_train, y_pred_train)
print(f"Training Accuracy: {accuracy_train * 100:.2f}%")

# Compute the testing accuracy
accuracy_test = accuracy_score(y_test, y_pred_test)
print(f"Testing Accuracy: {accuracy_test * 100:.2f}%")

print()

# Display precision, recall, F1-score and support
print(classification_report(
    y_test,
    y_pred_test,
    target_names=['Not Transported', 'Transported']
))

Training Accuracy: 79.11%
Testing Accuracy: 77.08%

                 precision    recall  f1-score   support

Not Transported       0.76      0.78      0.77       656
    Transported       0.78      0.76      0.77       666

       accuracy                           0.77      1322
      macro avg       0.77      0.77      0.77      1322
   weighted avg       0.77      0.77      0.77      1322



The AdaBoost classifier achieved a training accuracy of 79.11% and a testing accuracy of 77.08%. The small difference between the training and testing accuracies indicates that the model generalized well to unseen data and exhibited minimal overfitting. On the test set, the model achieved balanced precision, recall, and F1-scores of approximately 0.77 for both classes. Although the model demonstrated good generalization, its overall predictive performance was lower than that of the Random Forest and Gradient Boosting classifiers.

- XGBOOST

I will run it with random search, so we have optimised results

In [49]:
param_dist = {
    "n_estimators": [200, 250, 300, 350, 400],
    "learning_rate": [0.03, 0.05, 0.07, 0.1],
    "max_depth": [3, 4, 5],
    "min_child_weight": [2, 3, 4, 5],
    "subsample": [0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "gamma": [0, 0.1, 0.3, 0.5],
    "reg_alpha": [0, 0.1, 0.5, 1],
    "reg_lambda": [1, 5, 10, 15]
}

In [53]:
# Model Training: XGBoost Classifier
# ---------------------------------------------------

random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        random_state=SEED,
        eval_metric="logloss"
    ),
    param_distributions=param_dist,
    n_iter=50,          # Try 50 random combinations
    scoring="accuracy",
    cv=4,
    random_state=SEED,
    n_jobs=2
)

random_search.fit(X_train_final, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

Best Parameters: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 2, 'max_depth': 5, 'learning_rate': 0.03, 'gamma': 0.5, 'colsample_bytree': 1.0}
Best CV Score: 0.8109386828160484


In [54]:
best_xgb = random_search.best_estimator_

y_pred_train = best_xgb.predict(X_train_final)
y_pred_test = best_xgb.predict(X_test_final)

print(f"Training Accuracy: {accuracy_score(y_train, y_pred_train)*100:.2f}%")
print(f"Testing Accuracy: {accuracy_score(y_test, y_pred_test)*100:.2f}%")

print(classification_report(
    y_test,
    y_pred_test,
    target_names=["Not Transported", "Transported"]
))

Training Accuracy: 84.33%
Testing Accuracy: 78.97%
                 precision    recall  f1-score   support

Not Transported       0.80      0.77      0.78       656
    Transported       0.78      0.81      0.80       666

       accuracy                           0.79      1322
      macro avg       0.79      0.79      0.79      1322
   weighted avg       0.79      0.79      0.79      1322



The XGBoost classifier achieved a training accuracy of 84.33% and a testing accuracy of 78.97%. The relatively small difference between the training and testing accuracies indicates that the model generalized well to unseen data while exhibiting only limited overfitting. On the test set, the classifier achieved balanced precision, recall, and F1-scores of approximately 0.79 for both the Not Transported and Transported classes, demonstrating consistent predictive performance across both categories. Overall, the XGBoost classifier provided strong classification performance and was one of the best-performing ensemble models evaluated.

Which model is the best and why?

Among the ensemble models evaluated, the Gradient Boosting classifier achieved the best overall performance. It obtained the highest testing accuracy (79.05%) while maintaining a relatively small difference between the training accuracy (82.25%) and testing accuracy, indicating good generalization and limited overfitting. In addition, the model produced balanced precision, recall, and F1-scores for both the Transported and Not Transported classes, demonstrating consistent classification performance. Although the XGBoost classifier achieved comparable results, its testing accuracy (78.97%) was slightly lower. Therefore, the Gradient Boosting classifier was selected as the best-performing model because it provided the highest predictive accuracy while maintaining strong generalization to unseen data.